# Classifying emoji valence

**Models and datasets**

ML models: SVM, Random Forest, XGBoost

Embedding Models: emoji2vec, emojional

**Steps**
1. Run ML models on imnbalanced data
2. Run oversampling and undersampling parametrically. This will help us find the sweetspot of sample n per class that will optimize model performance
3. Use that sample n per class to run all ML models again. Compare performance of ML models and embeddings


In [ ]:
### Setup
import pandas as pd
import numpy as np
import seaborn as sns

from collections import Counter
import sklearn.model_selection as skm
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix, ConfusionMatrixDisplay, f1_score, recall_score
import matplotlib.pyplot as plt
from matplotlib.pyplot import subplots, cm
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedKFold
import scipy.stats as stats

from sklearn.metrics import roc_auc_score
import matplotlib.pyplot as plt
from sklearn.preprocessing import label_binarize
from sklearn.metrics import roc_curve, auc

from google.colab import files
import io

from collections import Counter
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler

In [ ]:
### upload both Emoji2Vecembeddings_overlap and Emojionalembeddings_overlap CSV files
uploaded = files.upload()

In [ ]:
df_e2v = pd.read_csv(io.StringIO(uploaded['Emoji2Vecembeddings_overlap.csv'].decode('utf-8')))
df_emo = pd.read_csv(io.StringIO(uploaded['Emojionalembeddings_overlap.csv'].decode('utf-8')))

display(df_e2v.head())
display(df_emo.head())

# Step 1: Running ML models on imbalanced data

In [ ]:
#Separate label and features

#emoji2vec
e2v_embedding_cols = [col for col in df_e2v.columns if col.startswith("vec")]

Xe2v = df_e2v[e2v_embedding_cols].values
ye2v = df_e2v["EmotionalValence"].values

print("emoji2vec", df_e2v["EmotionalValence"].value_counts())

#emojional
emo_embedding_cols = [col for col in df_emo.columns if col.startswith("vec")]

Xemo = df_emo[emo_embedding_cols].values
yemo = df_emo["EmotionalValence"].values

print("\nemojional", df_emo["EmotionalValence"].value_counts())

In [ ]:
#Train-test split (80/20)
Xe2v_train, Xe2v_test, ye2v_train, ye2v_test = train_test_split(
    Xe2v, ye2v,
    test_size=0.20,
    random_state=20, #randomness in how the data is split
    stratify=ye2v)      #to make both train and test sets to contain similar proportions

#Scale features
scaler = StandardScaler()
Xe2v_train = scaler.fit_transform(Xe2v_train)
Xe2v_test = scaler.transform(Xe2v_test)

#Train-test split (80/20)
Xemo_train, Xemo_test, yemo_train, yemo_test = train_test_split(
    Xemo, yemo,
    test_size=0.20,
    random_state=20,
    stratify=yemo)

#Scale features
scaler = StandardScaler()
Xemo_train = scaler.fit_transform(Xemo_train)
Xemo_test = scaler.transform(Xemo_test)

### (1) CV for SVM

In [ ]:
def run_svm_cross_validation(X, y, embedding_name, n_splits=10, random_state=42):
    print(f"--- Running SVM Cross-Validation for {embedding_name} ---")

    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)

    accuracies = []
    f1_macros = []
    roc_aucs = []
    f1_negatives = []
    f1_neutrals = []
    f1_positives = []

    all_y_true = [] # Initialize list to store true labels
    all_y_pred = [] # Initialize list to store predicted labels

    le = LabelEncoder()
    y_encoded = le.fit_transform(y)

    for fold, (train_index, test_index) in enumerate(skf.split(X, y_encoded)):
        X_train_fold, X_test_fold = X[train_index], X[test_index]
        y_train_fold, y_test_fold = y_encoded[train_index], y_encoded[test_index]

        scaler = StandardScaler()
        X_train_fold_scaled = scaler.fit_transform(X_train_fold)
        X_test_fold_scaled = scaler.transform(X_test_fold)

        model = SVC(
            kernel='rbf',
            C=1e5,
            gamma='scale',
            decision_function_shape='ovo',
            probability=True
        )

        model.fit(X_train_fold_scaled, y_train_fold)
        y_pred_fold = model.predict(X_test_fold_scaled)
        y_proba_fold = model.predict_proba(X_test_fold_scaled)

        all_y_true.append(y_test_fold) # Append true labels for the fold
        all_y_pred.append(y_pred_fold) # Append predicted labels for the fold

        acc = accuracy_score(y_test_fold, y_pred_fold)
        f1_macro = f1_score(y_test_fold, y_pred_fold, average='macro', zero_division=0)
        roc_auc = roc_auc_score(y_test_fold, y_proba_fold, multi_class='ovr', average='macro')

        report = classification_report(y_test_fold, y_pred_fold, output_dict=True, zero_division=0)
        f1_negatives.append(report['0']['f1-score'])
        f1_neutrals.append(report['1']['f1-score'])
        f1_positives.append(report['2']['f1-score'])

        accuracies.append(acc)
        f1_macros.append(f1_macro)
        roc_aucs.append(roc_auc)

        print(f"  Fold {fold+1}: Accuracy={acc:.3f}, Macro F1={f1_macro:.3f}, ROC AUC={roc_auc:.3f}, Neg F1={report['0']['f1-score']:.3f}, Neu F1={report['1']['f1-score']:.3f}, Pos F1={report['2']['f1-score']:.3f}")

    print(f"\n--- Average Results for {embedding_name} ---")
    print(f"  Average Accuracy: {np.mean(accuracies):.3f} (Std: {np.std(accuracies):.3f})")
    print(f"  Average Macro F1: {np.mean(f1_macros):.3f} (Std: {np.std(f1_macros):.3f})")
    print(f"  Average ROC AUC: {np.mean(roc_aucs):.3f} (Std: {np.std(roc_aucs):.3f})")
    print(f"  Average Negative F1: {np.mean(f1_negatives):.3f} (Std: {np.std(f1_negatives):.3f})")
    print(f"  Average Neutral F1: {np.mean(f1_neutrals):.3f} (Std: {np.std(f1_neutrals):.3f})")
    print(f"  Average Positive F1: {np.mean(f1_positives):.3f} (Std: {np.std(f1_positives):.3f})")
    print(f"--- Finished SVM Cross-Validation for {embedding_name} ---")

    # Concatenate all true and predicted labels
    all_y_true_concat = np.concatenate(all_y_true)
    all_y_pred_concat = np.concatenate(all_y_pred)

    return {'accuracies': accuracies, 'f1_macros': f1_macros, 'roc_aucs': roc_aucs, 'f1_negatives': f1_negatives, 'f1_neutrals': f1_neutrals, 'f1_positives': f1_positives, 'all_y_true': all_y_true_concat, 'all_y_pred': all_y_pred_concat}

# Run cross-validation for emoji2vec
svm_cv_e2v_results = run_svm_cross_validation(Xe2v, ye2v, "emoji2vec")

# Run cross-validation for emojional
svm_cv_emo_results = run_svm_cross_validation(Xemo, yemo, "emojional")

In [ ]:
### SVM imbalanced - confusion matrix

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
class_labels = ['Negative', 'Neutral', 'Positive']

# Plot for SVM with emoji2vec embeddings
ConfusionMatrixDisplay.from_predictions(
    svm_cv_e2v_results['all_y_true'],
    svm_cv_e2v_results['all_y_pred'],
    display_labels=class_labels,
    cmap=plt.cm.Blues,
    ax=axes[0]
)
axes[0].set_title('Confusion Matrix: SVM with emoji2vec embeddings')

# Plot for SVM with emojional embeddings
ConfusionMatrixDisplay.from_predictions(
    svm_cv_emo_results['all_y_true'],
    svm_cv_emo_results['all_y_pred'],
    display_labels=class_labels,
    cmap=plt.cm.Blues,
    ax=axes[1]
)
axes[1].set_title('Confusion Matrix: SVM with emojional embeddings')

plt.tight_layout()
plt.show()

### (2) CV for Random Forest

In [ ]:
from sklearn.model_selection import StratifiedKFold

def run_rf_cross_validation(X, y, embedding_name, n_splits=10, random_state=42):
    print(f"--- Running Random Forest Cross-Validation for {embedding_name} ---")

    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)

    accuracies = []
    f1_macros = []
    roc_aucs = []
    f1_negatives = []
    f1_neutrals = []
    f1_positives = []

    all_y_true = [] # Initialize list to store true labels
    all_y_pred = [] # Initialize list to store predicted labels

    le = LabelEncoder()
    y_encoded = le.fit_transform(y)

    for fold, (train_index, test_index) in enumerate(skf.split(X, y_encoded)):
        X_train_fold, X_test_fold = X[train_index], X[test_index]
        y_train_fold, y_test_fold = y_encoded[train_index], y_encoded[test_index]

        # Scale features within each fold to prevent data leakage
        scaler = StandardScaler()
        X_train_fold_scaled = scaler.fit_transform(X_train_fold)
        X_test_fold_scaled = scaler.transform(X_test_fold)

        model = RandomForestClassifier(
            n_estimators=100,
            max_depth=6,
            min_samples_leaf=5,
            random_state=random_state
        )

        model.fit(X_train_fold_scaled, y_train_fold)
        y_pred_fold = model.predict(X_test_fold_scaled)
        y_proba_fold = model.predict_proba(X_test_fold_scaled)

        all_y_true.append(y_test_fold) # Append true labels for the fold
        all_y_pred.append(y_pred_fold) # Append predicted labels for the fold

        acc = accuracy_score(y_test_fold, y_pred_fold)
        f1_macro = f1_score(y_test_fold, y_pred_fold, average='macro', zero_division=0)
        roc_auc = roc_auc_score(y_test_fold, y_proba_fold, multi_class='ovr', average='macro')

        report = classification_report(y_test_fold, y_pred_fold, output_dict=True, zero_division=0)
        f1_negatives.append(report['0']['f1-score'])
        f1_neutrals.append(report['1']['f1-score'])
        f1_positives.append(report['2']['f1-score'])

        accuracies.append(acc)
        f1_macros.append(f1_macro)
        roc_aucs.append(roc_auc)

        print(f"  Fold {fold+1}: Accuracy={acc:.3f}, Macro F1={f1_macro:.3f}, ROC AUC={roc_auc:.3f}, Neg F1={report['0']['f1-score']:.3f}, Neu F1={report['1']['f1-score']:.3f}, Pos F1={report['2']['f1-score']:.3f}")

    print(f"\n--- Average Results for {embedding_name} ---")
    print(f"  Average Accuracy: {np.mean(accuracies):.3f} (Std: {np.std(accuracies):.3f})")
    print(f"  Average Macro F1: {np.mean(f1_macros):.3f} (Std: {np.std(f1_macros):.3f})")
    print(f"  Average ROC AUC: {np.mean(roc_aucs):.3f} (Std: {np.std(roc_aucs):.3f})")
    print(f"  Average Negative F1: {np.mean(f1_negatives):.3f} (Std: {np.std(f1_negatives):.3f})")
    print(f"  Average Neutral F1: {np.mean(f1_neutrals):.3f} (Std: {np.std(f1_neutrals):.3f})")
    print(f"  Average Positive F1: {np.mean(f1_positives):.3f} (Std: {np.std(f1_positives):.3f})")
    print(f"--- Finished Random Forest Cross-Validation for {embedding_name} ---")

    # Concatenate all true and predicted labels
    all_y_true_concat = np.concatenate(all_y_true)
    all_y_pred_concat = np.concatenate(all_y_pred)

    return {'accuracies': accuracies, 'f1_macros': f1_macros, 'roc_aucs': roc_aucs, 'f1_negatives': f1_negatives, 'f1_neutrals': f1_neutrals, 'f1_positives': f1_positives, 'all_y_true': all_y_true_concat, 'all_y_pred': all_y_pred_concat}

# Run cross-validation for emoji2vec
rf_cv_e2v_results = run_rf_cross_validation(Xe2v, ye2v, "emoji2vec")

# Run cross-validation for emojional
rf_cv_emo_results = run_rf_cross_validation(Xemo, yemo, "emojional")

In [ ]:
### RF imbalanced - Confusion matrix

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
class_labels = ['Negative', 'Neutral', 'Positive']

# Plot for Random Forest with emoji2vec embeddings
ConfusionMatrixDisplay.from_predictions(
    rf_cv_e2v_results['all_y_true'],
    rf_cv_e2v_results['all_y_pred'],
    display_labels=class_labels,
    cmap=plt.cm.Blues,
    ax=axes[0]
)
axes[0].set_title('Confusion Matrix: Random Forest with emoji2vec embeddings')

# Plot for Random Forest with emojional embeddings
ConfusionMatrixDisplay.from_predictions(
    rf_cv_emo_results['all_y_true'],
    rf_cv_emo_results['all_y_pred'],
    display_labels=class_labels,
    cmap=plt.cm.Blues,
    ax=axes[1]
)
axes[1].set_title('Confusion Matrix: Random Forest with emojional embeddings')

plt.tight_layout()
plt.show()

### (3) CV for XGBoost

In [ ]:
def run_xgb_cross_validation(X, y, embedding_name, n_splits=10, random_state=42):
    print(f"--- Running XGBoost Cross-Validation for {embedding_name} ---")

    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)

    accuracies = []
    f1_macros = []
    roc_aucs = []
    f1_negatives = []
    f1_neutrals = []
    f1_positives = []

    all_y_true = [] # Initialize list to store true labels
    all_y_pred = [] # Initialize list to store predicted labels

    # XGBoost requires numerical labels (0, 1, 2) which were already encoded before.
    # However, to ensure consistency and reusability, we re-encode if necessary.
    le = LabelEncoder()
    y_encoded = le.fit_transform(y)
    num_classes = len(np.unique(y_encoded))

    for fold, (train_index, test_index) in enumerate(skf.split(X, y_encoded)):
        X_train_fold, X_test_fold = X[train_index], X[test_index]
        y_train_fold, y_test_fold = y_encoded[train_index], y_encoded[test_index]

        scaler = StandardScaler()
        X_train_fold_scaled = scaler.fit_transform(X_train_fold)
        X_test_fold_scaled = scaler.transform(X_test_fold)

        model = XGBClassifier(
            objective="multi:softprob",
            num_class=num_classes,
            n_estimators=200,
            learning_rate=0.1,
            max_depth=6,
            random_state=random_state,
            eval_metric="mlogloss"
        )

        model.fit(X_train_fold_scaled, y_train_fold)
        y_pred_fold = model.predict(X_test_fold_scaled)
        y_proba_fold = model.predict_proba(X_test_fold_scaled)

        all_y_true.append(y_test_fold) # Append true labels for the fold
        all_y_pred.append(y_pred_fold) # Append predicted labels for the fold

        acc = accuracy_score(y_test_fold, y_pred_fold)
        f1_macro = f1_score(y_test_fold, y_pred_fold, average='macro', zero_division=0)
        roc_auc = roc_auc_score(y_test_fold, y_proba_fold, multi_class='ovr', average='macro')

        report = classification_report(y_test_fold, y_pred_fold, output_dict=True, zero_division=0)
        f1_negatives.append(report['0']['f1-score'])
        f1_neutrals.append(report['1']['f1-score'])
        f1_positives.append(report['2']['f1-score'])

        accuracies.append(acc)
        f1_macros.append(f1_macro)
        roc_aucs.append(roc_auc)

        print(f"  Fold {fold+1}: Accuracy={acc:.3f}, Macro F1={f1_macro:.3f}, ROC AUC={roc_auc:.3f}, Neg F1={report['0']['f1-score']:.3f}, Neu F1={report['1']['f1-score']:.3f}, Pos F1={report['2']['f1-score']:.3f}")

    print(f"\n--- Average Results for {embedding_name} ---")
    print(f"  Average Accuracy: {np.mean(accuracies):.3f} (Std: {np.std(accuracies):.3f})")
    print(f"  Average Macro F1: {np.mean(f1_macros):.3f} (Std: {np.std(f1_macros):.3f})")
    print(f"  Average ROC AUC: {np.mean(roc_aucs):.3f} (Std: {np.std(roc_aucs):.3f})")
    print(f"  Average Negative F1: {np.mean(f1_negatives):.3f} (Std: {np.std(f1_negatives):.3f})")
    print(f"  Average Neutral F1: {np.mean(f1_neutrals):.3f} (Std: {np.std(f1_neutrals):.3f})")
    print(f"  Average Positive F1: {np.mean(f1_positives):.3f} (Std: {np.std(f1_positives):.3f})")
    print(f"--- Finished XGBoost Cross-Validation for {embedding_name} ---")

    # Concatenate all true and predicted labels
    all_y_true_concat = np.concatenate(all_y_true)
    all_y_pred_concat = np.concatenate(all_y_pred)

    return {'accuracies': accuracies, 'f1_macros': f1_macros, 'roc_aucs': roc_aucs, 'f1_negatives': f1_negatives, 'f1_neutrals': f1_neutrals, 'f1_positives': f1_positives, 'all_y_true': all_y_true_concat, 'all_y_pred': all_y_pred_concat}

# Run cross-validation for emoji2vec
xgb_cv_e2v_results = run_xgb_cross_validation(Xe2v, ye2v, "emoji2vec")

# Run cross-validation for emojional
xgb_cv_emo_results = run_xgb_cross_validation(Xemo, yemo, "emojional")

In [ ]:
### XGBoost imbalanced - Confusion matrix

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
class_labels = ['Negative', 'Neutral', 'Positive']

# Plot for XGBoost with emoji2vec embeddings
ConfusionMatrixDisplay.from_predictions(
    xgb_cv_e2v_results['all_y_true'],
    xgb_cv_e2v_results['all_y_pred'],
    display_labels=class_labels,
    cmap=plt.cm.Blues,
    ax=axes[0]
)
axes[0].set_title('Confusion Matrix: XGBoost with emoji2vec embeddings')

# Plot for XGBoost with emojional embeddings
ConfusionMatrixDisplay.from_predictions(
    xgb_cv_emo_results['all_y_true'],
    xgb_cv_emo_results['all_y_pred'],
    display_labels=class_labels,
    cmap=plt.cm.Blues,
    ax=axes[1]
)
axes[1].set_title('Confusion Matrix: XGBoost with emojional embeddings')

plt.tight_layout()
plt.show()

## T-test to compare between models

In [ ]:
def run_t_tests(results_dict_1, results_dict_2, name_1, name_2, metric_context):
    print(f"\n--- T-Tests for {metric_context}: {name_1} vs {name_2} ---")

    # Accuracy comparison
    acc_1 = results_dict_1['accuracies']
    acc_2 = results_dict_2['accuracies']
    t_stat_acc, p_value_acc = stats.ttest_rel(acc_1, acc_2)
    print(f"Accuracy (p-value): {p_value_acc:.4f}")

    # Macro F1 comparison
    f1_1 = results_dict_1['f1_macros']
    f1_2 = results_dict_2['f1_macros']
    t_stat_f1, p_value_f1 = stats.ttest_rel(f1_1, f1_2)
    print(f"Macro F1 (p-value): {p_value_f1:.4f}")

# --- Comparing Models within each Embedding ---
print("\n*** Comparing Models within emoji2vec Embeddings ***")
run_t_tests(svm_cv_e2v_results, rf_cv_e2v_results, "SVM", "Random Forest", "emoji2vec")
run_t_tests(svm_cv_e2v_results, xgb_cv_e2v_results, "SVM", "XGBoost", "emoji2vec")
run_t_tests(rf_cv_e2v_results, xgb_cv_e2v_results, "Random Forest", "XGBoost", "emoji2vec")

print("\n*** Comparing Models within emojional Embeddings ***")
run_t_tests(svm_cv_emo_results, rf_cv_emo_results, "SVM", "Random Forest", "emojional")
run_t_tests(svm_cv_emo_results, xgb_cv_emo_results, "SVM", "XGBoost", "emojional")
run_t_tests(rf_cv_emo_results, xgb_cv_emo_results, "Random Forest", "XGBoost", "emojional")

# --- Comparing Embeddings for each Model ---
print("\n*** Comparing Embeddings for each Model ***")
run_t_tests(svm_cv_e2v_results, svm_cv_emo_results, "emoji2vec", "emojional", "SVM")
run_t_tests(rf_cv_e2v_results, rf_cv_emo_results, "emoji2vec", "emojional", "Random Forest")
run_t_tests(xgb_cv_e2v_results, xgb_cv_emo_results, "emoji2vec", "emojional", "XGBoost")

## Bootstrapping

In [ ]:
def run_bootstrap_experiment(model_type, X_original, y_original, embedding_name, n_iterations=100, random_state=42):
    print(f"--- Running Bootstrapping for {model_type} with {embedding_name} ---")

    accuracies = []
    f1_macros = []
    f1_negatives = []
    f1_neutrals = []
    f1_positives = []

    # Ensure y is encoded for consistency with models, especially XGBoost
    le = LabelEncoder()
    y_encoded = le.fit_transform(y_original)
    num_classes = len(np.unique(y_encoded))
    class_labels = sorted(le.classes_)

    for i in range(n_iterations):
        # Set a seed for reproducibility within each iteration
        np.random.seed(random_state + i)

        # Generate stratified bootstrap indices by sampling with replacement within each class
        stratified_bootstrap_indices = []
        unique_classes = np.unique(y_encoded)
        for c in unique_classes:
            class_indices = np.where(y_encoded == c)[0]
            n_samples_for_class = len(class_indices)
            # Sample with replacement from indices of the current class
            resampled_class_indices = np.random.choice(class_indices, size=n_samples_for_class, replace=False)
            stratified_bootstrap_indices.extend(resampled_class_indices)

        # Convert to numpy array and shuffle to mix samples from different classes
        stratified_bootstrap_indices = np.array(stratified_bootstrap_indices)
        np.random.shuffle(stratified_bootstrap_indices) # Shuffle to mix classes

        X_boot = X_original[stratified_bootstrap_indices]
        y_boot = y_encoded[stratified_bootstrap_indices]

        # Split the bootstrapped data into training and testing sets (stratified split here)
        X_train_boot, X_test_boot, y_train_boot, y_test_boot = train_test_split(
            X_boot, y_boot, test_size=0.20, random_state=random_state + i, stratify=y_boot)

        # Scale features
        scaler = StandardScaler()
        X_train_boot_scaled = scaler.fit_transform(X_train_boot)
        X_test_boot_scaled = scaler.transform(X_test_boot)

        model = None
        if model_type == 'SVM':
            model = SVC(
                kernel='rbf',
                C=1e5,
                gamma='scale',
                decision_function_shape='ovo',
                probability=True,
                random_state=random_state
            )
        elif model_type == 'Random Forest':
            model = RandomForestClassifier(
                n_estimators=100,
                max_depth=6,
                min_samples_leaf=5,
                random_state=random_state
            )
        elif model_type == 'XGBoost':
            model = XGBClassifier(
                objective="multi:softprob",
                num_class=num_classes,
                n_estimators=200,
                learning_rate=0.1,
                max_depth=6,
                random_state=random_state,
                eval_metric="mlogloss"
            )
        else:
            raise ValueError("Invalid model_type specified.")

        model.fit(X_train_boot_scaled, y_train_boot)
        y_pred_boot = model.predict(X_test_boot_scaled)
        y_proba_boot = model.predict_proba(X_test_boot_scaled)

        acc = accuracy_score(y_test_boot, y_pred_boot)
        f1_macro = f1_score(y_test_boot, y_pred_boot, average='macro', zero_division=0)
        # ROC AUC for multiclass requires probability predictions
        # Roc_auc calculation if needed here for each fold using y_proba_boot

        report = classification_report(y_test_boot, y_pred_boot, output_dict=True, zero_division=0)

        accuracies.append(acc)
        f1_macros.append(f1_macro)
        f1_negatives.append(report[str(le.transform([class_labels[0]])[0])]['f1-score'])
        f1_neutrals.append(report[str(le.transform([class_labels[1]])[0])]['f1-score'])
        f1_positives.append(report[str(le.transform([class_labels[2]])[0])]['f1-score'])

        if (i + 1) % 10 == 0: # Print progress every 10 iterations
            print(f"  Iteration {i+1}/{n_iterations}: Acc={acc:.3f}, Macro F1={f1_macro:.3f}")

    print(f"--- Finished Bootstrapping for {model_type} with {embedding_name} ---")
    return {
        'accuracies': np.array(accuracies),
        'f1_macros': np.array(f1_macros),
        'f1_negatives': np.array(f1_negatives),
        'f1_neutrals': np.array(f1_neutrals),
        'f1_positives': np.array(f1_positives),
        'le': le
    }

In [ ]:
# Number of bootstrap iterations
N_BOOTSTRAP_ITERATIONS = 100

# Dictionary to store bootstrap results
bootstrap_results = {}

# --- emoji2vec Embeddings ---
print("\n*** Running Bootstrapping for emoji2vec Embeddings ***")
bootstrap_results['SVM_e2v'] = run_bootstrap_experiment('SVM', Xe2v, ye2v, "emoji2vec", n_iterations=N_BOOTSTRAP_ITERATIONS)
bootstrap_results['RF_e2v'] = run_bootstrap_experiment('Random Forest', Xe2v, ye2v, "emoji2vec", n_iterations=N_BOOTSTRAP_ITERATIONS)
bootstrap_results['XGB_e2v'] = run_bootstrap_experiment('XGBoost', Xe2v, ye2v, "emoji2vec", n_iterations=N_BOOTSTRAP_ITERATIONS)

# --- emojional Embeddings ---
print("\n*** Running Bootstrapping for emojional Embeddings ***")
bootstrap_results['SVM_emo'] = run_bootstrap_experiment('SVM', Xemo, yemo, "emojional", n_iterations=N_BOOTSTRAP_ITERATIONS)
bootstrap_results['RF_emo'] = run_bootstrap_experiment('Random Forest', Xemo, yemo, "emojional", n_iterations=N_BOOTSTRAP_ITERATIONS)
bootstrap_results['XGB_emo'] = run_bootstrap_experiment('XGBoost', Xemo, yemo, "emojional", n_iterations=N_BOOTSTRAP_ITERATIONS)

print("\n*** All Bootstrapping experiments completed ***")


In [ ]:
def calculate_confidence_interval(data, confidence=0.95):
    a = np.array(data)
    p = ((1.0 - confidence) / 2.0) * 100
    lower = np.percentile(a, p)
    p = (confidence + (1.0 - confidence) / 2.0) * 100
    upper = np.percentile(a, p)
    return lower, upper


# Prepare data for plotting
plot_data = []

for key, res in bootstrap_results.items():
    model_name = key.split('_')[0]
    embedding_name = 'emoji2vec' if 'e2v' in key else 'emojional'

    # Accuracy
    mean_acc = np.mean(res['accuracies'])
    lower_acc, upper_acc = calculate_confidence_interval(res['accuracies'])
    plot_data.append({
        'Metric': 'Accuracy',
        'Model': model_name,
        'Embedding': embedding_name,
        'Mean': mean_acc,
        'Lower_CI': lower_acc,
        'Upper_CI': upper_acc
    })

    # Macro F1
    mean_f1 = np.mean(res['f1_macros'])
    lower_f1, upper_f1 = calculate_confidence_interval(res['f1_macros'])
    plot_data.append({
        'Metric': 'Macro F1',
        'Model': model_name,
        'Embedding': embedding_name,
        'Mean': mean_f1,
        'Lower_CI': lower_f1,
        'Upper_CI': upper_f1
    })

plot_df = pd.DataFrame(plot_data)
display(plot_df)


In [ ]:
metrics = plot_df['Metric'].unique()

for metric in metrics:
    df_subset = plot_df[plot_df['Metric'] == metric]

    models = df_subset['Model'].unique()
    embeddings = ["emoji2vec", "emojional"]
    y = np.arange(len(models))
    offset = 0.2

    plt.figure(figsize=(8, 5))

    for i, emb in enumerate(embeddings):
        emb_data = df_subset[df_subset['Embedding'] == emb]

        means = emb_data['Mean'].values
        lower = emb_data['Lower_CI'].values
        upper = emb_data['Upper_CI'].values

        y_positions = y + (len(embeddings)/2 - i - 0.5) * offset

        # Horizontal CI lines
        for j in range(len(means)):
            plt.hlines(y=y_positions[j], xmin=lower[j], xmax=upper[j], color=plt.cm.Paired(i))

        # Mean dots
        plt.scatter(means, y_positions, label=emb, s=60, color=plt.cm.Paired(i))

    plt.yticks(y, models)
    plt.xlabel(metric)
    plt.ylabel("Model")
    plt.title(f"{metric} with 95% Confidence Intervals (Imbalanced Data)")

    plt.xlim(0, 1)
    plt.grid(False)

    if metric == "Accuracy":
        plt.axvline(0.6349, linestyle='--', linewidth=1, color='gray', label='Baseline Accuracy')
    elif metric == "Macro F1":
        plt.axvline(0.3333, linestyle='--', linewidth=1, color='gray', label='Baseline Macro F1')

    plt.legend(title="Embedding", loc="upper left")
    plt.tight_layout()
    plt.show()

# Step 2 : Paramatrically run oversampling and undersampling

In [ ]:
def resample_to_target(X, y, target_n, random_state=42):

    counts = Counter(y)

    over_dict = {c: target_n for c in counts if counts[c] < target_n}
    under_dict = {c: target_n for c in counts if counts[c] > target_n}

    X_res, y_res = X, y

    if len(over_dict) > 0:
        sm = SMOTE(sampling_strategy=over_dict, random_state=random_state)
        X_res, y_res = sm.fit_resample(X_res, y_res)

    if len(under_dict) > 0:
        rus = RandomUnderSampler(sampling_strategy=under_dict, random_state=random_state)
        X_res, y_res = rus.fit_resample(X_res, y_res)

    return X_res, y_res

### (1) SVM

In [ ]:
def run_svm_parametric_experiment(X_train, y_train, X_test, y_test, embedding_name):
    classes = sorted(np.unique(y_train))

    sample_range = np.linspace(1, 500, 20).astype(int)

    accuracies = []
    f1_scores = []

    f1_negative = []
    f1_neutral = []
    f1_positive = []

    print(f"--- Running SVM parametric experiment for {embedding_name} ---")
    for n in sample_range:

        X_bal, y_bal = resample_to_target(X_train, y_train, n)

        model = SVC(
            kernel='rbf',
            C=1e5,
            gamma='scale',
            decision_function_shape='ovo',
            probability=True, # probability=True is needed for ROC AUC later, good to include here
            ) # Set random_state for reproducibility

        model.fit(X_bal, y_bal)

        y_pred = model.predict(X_test)

        acc = accuracy_score(y_test, y_pred)
        f1_macro = f1_score(y_test, y_pred, average='macro', zero_division=0)

        report = classification_report(y_test, y_pred, output_dict=True, zero_division=0)

        accuracies.append(acc)
        f1_scores.append(f1_macro)

        # Accessing report using string representation of numeric classes
        f1_negative.append(report[str(classes[0])]['f1-score'])
        f1_neutral.append(report[str(classes[1])]['f1-score'])
        f1_positive.append(report[str(classes[2])]['f1-score'])

        print(
            f"  {n} samples/class ({embedding_name}) -> "
            f"Acc: {acc:.3f}, F1 (Macro): {f1_macro:.3f}, "
            f"Negative F1: {report[str(classes[0])]['f1-score']:.3f}, "
            f"Neutral F1: {report[str(classes[1])]['f1-score']:.3f}, "
            f"Positive F1: {report[str(classes[2])]['f1-score']:.3f}")
    print(f"--- Finished SVM parametric experiment for {embedding_name} ---")
    return accuracies, f1_scores, f1_negative, f1_neutral, f1_positive

In [ ]:
svm_e2v_accuracies, svm_e2v_f1_scores, svm_e2v_f1_negative, svm_e2v_f1_neutral, svm_e2v_f1_positive = run_svm_parametric_experiment(Xe2v_train, ye2v_train, Xe2v_test, ye2v_test, "emoji2vec")
svm_emo_accuracies, svm_emo_f1_scores, svm_emo_f1_negative, svm_emo_f1_neutral, svm_emo_f1_positive = run_svm_parametric_experiment(Xemo_train, yemo_train, Xemo_test, yemo_test, "emojional")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 6))
plt.rcParams.update({'font.size': 16})
sample_range = np.linspace(1, 500, 20).astype(int)

# Plot for emoji2vec
axes[0].plot(sample_range, svm_e2v_accuracies, marker='o', label="Accuracy", color='red')
axes[0].plot(sample_range, svm_e2v_f1_scores, marker='o', label="Macro F1", color='orange')
axes[0].plot(sample_range, svm_e2v_f1_negative, linestyle='--', marker='o', label="F1 Negative", color='forestgreen')
axes[0].plot(sample_range, svm_e2v_f1_neutral, linestyle='--', marker='o', label="F1 Neutral", color='limegreen')
axes[0].plot(sample_range, svm_e2v_f1_positive, linestyle='--', marker='o', label="F1 Positive", color='yellowgreen')
axes[0].set_xlabel("Samples per class after resampling")
axes[0].set_ylabel("Score")
axes[0].set_title("Class Balancing Size on SVM Performance - emoji2vec")
axes[0].legend()
axes[0].grid(alpha=0.3)
axes[0].set_ylim(0, 1) # Set y-axis limit

# Plot for emojional
axes[1].plot(sample_range, svm_emo_accuracies, marker='o', label="Accuracy", color = 'red')
axes[1].plot(sample_range, svm_emo_f1_scores, marker='o', label="Macro F1", color = 'orange')
axes[1].plot(sample_range, svm_emo_f1_negative, linestyle='--', marker='o', label="F1 Negative", color='forestgreen')
axes[1].plot(sample_range, svm_emo_f1_neutral, linestyle='--', marker='o', label="F1 Neutral", color='limegreen')
axes[1].plot(sample_range, svm_emo_f1_positive, linestyle='--', marker='o', label="F1 Positive", color='yellowgreen')
axes[1].set_xlabel("Samples per class after resampling")
axes[1].set_ylabel("Score")
axes[1].set_title("Class Balancing Size on SVM Performance - emojional")
axes[1].legend()
axes[1].grid(alpha=0.3)
axes[1].set_ylim(0, 1) # Set y-axis limit

plt.tight_layout()
plt.savefig("svm_parametric_experiment_plot.png")
plt.show()

### (2) Random Forest

In [ ]:
def run_rf_parametric_experiment(X_train, y_train, X_test, y_test, embedding_name, random_state=42):
    classes = sorted(np.unique(y_train))

    sample_range = np.linspace(1, 500, 20).astype(int)

    accuracies = []
    f1_scores = []

    f1_negative = []
    f1_neutral = []
    f1_positive = []

    print(f"--- Running Random Forest parametric experiment for {embedding_name} ---")
    for n in sample_range:

        X_bal, y_bal = resample_to_target(X_train, y_train, n, random_state=random_state)

        model = RandomForestClassifier(
            n_estimators=100,
            max_depth=6,
            min_samples_leaf=5,
            random_state=random_state)

        model.fit(X_bal, y_bal)

        y_pred = model.predict(X_test)

        acc = accuracy_score(y_test, y_pred)
        f1_macro = f1_score(y_test, y_pred, average='macro', zero_division=0)

        report = classification_report(y_test, y_pred, output_dict=True, zero_division=0)

        accuracies.append(acc)
        f1_scores.append(f1_macro)

        # Accessing report using string representation of numeric classes
        f1_negative.append(report[str(classes[0])]['f1-score'])
        f1_neutral.append(report[str(classes[1])]['f1-score'])
        f1_positive.append(report[str(classes[2])]['f1-score'])

        print(
            f"  {n} samples/class ({embedding_name}) -> "
            f"Acc: {acc:.3f}, F1 (Macro): {f1_macro:.3f}, "
            f"Negative F1: {report[str(classes[0])]['f1-score']:.3f}, "
            f"Neutral F1: {report[str(classes[1])]['f1-score']:.3f}, "
            f"Positive F1: {report[str(classes[2])]['f1-score']:.3f}")
    print(f"--- Finished Random Forest parametric experiment for {embedding_name} ---")
    return accuracies, f1_scores, f1_negative, f1_neutral, f1_positive

In [ ]:
rf_e2v_accuracies, rf_e2v_f1_scores, rf_e2v_f1_negative, rf_e2v_f1_neutral, rf_e2v_f1_positive = run_rf_parametric_experiment(Xe2v_train, ye2v_train, Xe2v_test, ye2v_test, "emoji2vec", random_state=42)
rf_emo_accuracies, rf_emo_f1_scores, rf_emo_f1_negative, rf_emo_f1_neutral, rf_emo_f1_positive = run_rf_parametric_experiment(Xemo_train, yemo_train, Xemo_test, yemo_test, "emojional", random_state=42)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 6))
plt.rcParams.update({'font.size': 16})
sample_range = np.linspace(1, 500, 20).astype(int)

# Plot for emoji2vec
axes[0].plot(sample_range, rf_e2v_accuracies, marker='o', label="Accuracy", color='red')
axes[0].plot(sample_range, rf_e2v_f1_scores, marker='o', label="Macro F1", color='orange')
axes[0].plot(sample_range, rf_e2v_f1_negative, linestyle='--', marker='o', label="F1 Negative", color='forestgreen')
axes[0].plot(sample_range, rf_e2v_f1_neutral, linestyle='--', marker='o', label="F1 Neutral", color='limegreen')
axes[0].plot(sample_range, rf_e2v_f1_positive, linestyle='--', marker='o', label="F1 Positive", color='yellowgreen')
axes[0].set_xlabel("Samples per class after resampling")
axes[0].set_ylabel("Score")
axes[0].set_title("Class Balancing Size on RF Performance - emoji2vec")
axes[0].legend()
axes[0].grid(alpha=0.3)
axes[0].set_ylim(0, 1) # Set y-axis limit

# Plot for emojional
axes[1].plot(sample_range, rf_emo_accuracies, marker='o', label="Accuracy", color = 'red')
axes[1].plot(sample_range, rf_emo_f1_scores, marker='o', label="Macro F1", color = 'orange')
axes[1].plot(sample_range, rf_emo_f1_negative, linestyle='--', marker='o', label="F1 Negative", color='forestgreen')
axes[1].plot(sample_range, rf_emo_f1_neutral, linestyle='--', marker='o', label="F1 Neutral", color='limegreen')
axes[1].plot(sample_range, rf_emo_f1_positive, linestyle='--', marker='o', label="F1 Positive", color='yellowgreen')
axes[1].set_xlabel("Samples per class after resampling")
axes[1].set_ylabel("Score")
axes[1].set_title("Class Balancing Size on RF Performance - emojional")
axes[1].legend()
axes[1].grid(alpha=0.3)
axes[1].set_ylim(0, 1) # Set y-axis limit

plt.tight_layout()
plt.savefig("rf_parametric_experiment_plot.png")
plt.show()

### (3) XGBoost



In [ ]:
#convert labels to 0, 1, 2
le = LabelEncoder()

# Fit the LabelEncoder on all unique labels to ensure consistent mapping
all_labels = np.unique(np.concatenate((ye2v, yemo)))
le.fit(all_labels)

# Transform ye2v_train and ye2v_test
ye2v_train_encoded = le.transform(ye2v_train)
ye2v_test_encoded = le.transform(ye2v_test)

# Transform yemo_train and yemo_test
yemo_train_encoded = le.transform(yemo_train)
yemo_test_encoded = le.transform(yemo_test)

print("Original ye2v_train labels:", Counter(ye2v_train))
print("Encoded ye2v_train labels:", Counter(ye2v_train_encoded))
print("Label mapping:", list(le.classes_))


In [ ]:
def run_xgb_parametric_experiment(X_train, y_train, X_test, y_test, embedding_name, random_state=42):
    classes = sorted(np.unique(y_train))

    sample_range = np.linspace(1, 500, 20).astype(int)

    accuracies = []
    f1_scores = []

    f1_negative = []
    f1_neutral = []
    f1_positive = []

    print(f"--- Running XGBoost parametric experiment for {embedding_name} ---")
    for n in sample_range:

        X_bal, y_bal = resample_to_target(X_train, y_train, n, random_state=random_state)

        model = XGBClassifier(
            objective="multi:softprob",
            num_class=3,
            n_estimators=200,
            learning_rate=0.1,
            max_depth=6,
            random_state=random_state,
            eval_metric="mlogloss")

        model.fit(X_bal, y_bal)

        y_pred = model.predict(X_test)

        acc = accuracy_score(y_test, y_pred)
        f1_macro = f1_score(y_test, y_pred, average='macro', zero_division=0)

        report = classification_report(y_test, y_pred, output_dict=True, zero_division=0)

        accuracies.append(acc)
        f1_scores.append(f1_macro)

        # Accessing report using string representation of numeric classes
        f1_negative.append(report[str(classes[0])]['f1-score'])
        f1_neutral.append(report[str(classes[1])]['f1-score'])
        f1_positive.append(report[str(classes[2])]['f1-score'])

        print(
            f"  {n} samples/class ({embedding_name}) -> "
            f"Acc: {acc:.3f}, F1 (Macro): {f1_macro:.3f}, "
            f"Negative F1: {report[str(classes[0])]['f1-score']:.3f}, "
            f"Neutral F1: {report[str(classes[1])]['f1-score']:.3f}, "
            f"Positive F1: {report[str(classes[2])]['f1-score']:.3f}")
    print(f"--- Finished XGBoost parametric experiment for {embedding_name} ---")
    return accuracies, f1_scores, f1_negative, f1_neutral, f1_positive

In [ ]:
xgb_e2v_accuracies, xgb_e2v_f1_scores, xgb_e2v_f1_negative, xgb_e2v_f1_neutral, xgb_e2v_f1_positive = run_xgb_parametric_experiment(Xe2v_train, ye2v_train_encoded, Xe2v_test, ye2v_test_encoded, "emoji2vec", random_state=42)
xgb_emo_accuracies, xgb_emo_f1_scores, xgb_emo_f1_negative, xgb_emo_f1_neutral, xgb_emo_f1_positive = run_xgb_parametric_experiment(Xemo_train, yemo_train_encoded, Xemo_test, yemo_test_encoded, "emojional", random_state=42)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 6))
plt.rcParams.update({'font.size': 16})
sample_range = np.linspace(1, 500, 20).astype(int)

# Plot for emoji2vec
axes[0].plot(sample_range, xgb_e2v_accuracies, marker='o', label="Accuracy", color='red')
axes[0].plot(sample_range, xgb_e2v_f1_scores, marker='o', label="Macro F1", color='orange')
axes[0].plot(sample_range, xgb_e2v_f1_negative, linestyle='--', marker='o', label="F1 Negative", color='forestgreen')
axes[0].plot(sample_range, xgb_e2v_f1_neutral, linestyle='--', marker='o', label="F1 Neutral", color='limegreen')
axes[0].plot(sample_range, xgb_e2v_f1_positive, linestyle='--', marker='o', label="F1 Positive", color='yellowgreen')
axes[0].set_xlabel("Samples per class after resampling")
axes[0].set_ylabel("Score")
axes[0].set_title("Class Balancing Size on XGBoost Performance - emoji2vec")
axes[0].legend()
axes[0].grid(alpha=0.3)
axes[0].set_ylim(0, 1) # Set y-axis limit

# Plot for emojional
axes[1].plot(sample_range, xgb_emo_accuracies, marker='o', label="Accuracy", color = 'red')
axes[1].plot(sample_range, xgb_emo_f1_scores, marker='o', label="Macro F1", color = 'orange')
axes[1].plot(sample_range, xgb_emo_f1_negative, linestyle='--', marker='o', label="F1 Negative", color='forestgreen')
axes[1].plot(sample_range, xgb_emo_f1_neutral, linestyle='--', marker='o', label="F1 Neutral", color='limegreen')
axes[1].plot(sample_range, xgb_emo_f1_positive, linestyle='--', marker='o', label="F1 Positive", color='yellowgreen')
axes[1].set_xlabel("Samples per class after resampling")
axes[1].set_ylabel("Score")
axes[1].set_title("Class Balancing Size on XGBoost Performance - emojional")
axes[1].legend()
axes[1].grid(alpha=0.3)
axes[1].set_ylim(0, 1) # Set y-axis limit

plt.tight_layout()
plt.savefig("xgb_parametric_experiment_plot.png")
plt.show()

### Sweetspot? n=158 to 184/class

Looking at Macro F1 is more accurate. We see on the plots that around n=150-200, Macro F1 improved while maintaining relatively good balance of individual class. When n > 150-200, neutral class started to suffer.

Take the average: (158+184)/2 = 171

# Step 3: Running all ML models again with n=171



## 10-fold CV

In [ ]:
print(f'accuracy baseline: {(433/(433+182+67))*100:.2f}%')
print(f'macro f1 baseline: {((433/(433+182+67))*100+(182/(433+182+67))*100+(67/(433+182+67))*100)/3:.2f}%')

print(f'\nf1 baseline for positive: {(433/(433+182+67))*100:.2f}%')
print(f'f1 baseline for neutral: {(182/(433+182+67))*100:.2f}%')
print(f'f1 baseline for negative: {(67/(433+182+67))*100:.2f}%')

In [ ]:
def run_cv_experiment(model_type, X, y, embedding_name, target_n, n_splits=10, random_state=42):
    print(f"--- Running {model_type} Cross-Validation for {embedding_name} with target_n={target_n} ---")

    accuracies = []
    f1_macros = []
    roc_aucs = []
    f1_negatives = []
    f1_neutrals = []
    f1_positives = []

    all_y_true = []
    all_y_pred = []

    le = LabelEncoder()
    y_encoded = le.fit_transform(y)
    num_classes = len(np.unique(y_encoded))
    class_labels = sorted(le.classes_)

    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)

    for fold, (train_index, test_index) in enumerate(skf.split(X, y_encoded)):
        X_train_fold, X_test_fold = X[train_index], X[test_index]
        y_train_fold, y_test_fold = y_encoded[train_index], y_encoded[test_index]

        scaler = StandardScaler()
        X_train_fold_scaled = scaler.fit_transform(X_train_fold)
        X_test_fold_scaled = scaler.transform(X_test_fold)

        # Resample the training data
        X_bal, y_bal = resample_to_target(X_train_fold_scaled, y_train_fold, target_n, random_state=random_state)

        model = None
        if model_type == 'SVM':
            model = SVC(
                kernel='rbf',
                C=1e5,
                gamma='scale',
                decision_function_shape='ovo',
                probability=True)

        elif model_type == 'Random Forest':
            model = RandomForestClassifier(
                n_estimators=100,
                max_depth=6,
                min_samples_leaf=5,
                random_state=random_state)

        elif model_type == 'XGBoost':
            model = XGBClassifier(
                objective="multi:softprob",
                num_class=num_classes,
                n_estimators=200,
                learning_rate=0.1,
                max_depth=6,
                random_state=random_state,
                eval_metric="mlogloss")
        else:
            raise ValueError("Invalid model_type specified. Choose 'SVM', 'Random Forest', or 'XGBoost'.")

        model.fit(X_bal, y_bal)
        y_pred_fold = model.predict(X_test_fold_scaled)
        y_proba_fold = model.predict_proba(X_test_fold_scaled)

        all_y_true.append(y_test_fold)
        all_y_pred.append(y_pred_fold)

        acc = accuracy_score(y_test_fold, y_pred_fold)
        f1_macro = f1_score(y_test_fold, y_pred_fold, average='macro', zero_division=0)
        roc_auc = roc_auc_score(y_test_fold, y_proba_fold, multi_class='ovr', average='macro')

        report = classification_report(y_test_fold, y_pred_fold, output_dict=True, zero_division=0)
        f1_negatives.append(report[str(le.transform([class_labels[0]])[0])]['f1-score'])
        f1_neutrals.append(report[str(le.transform([class_labels[1]])[0])]['f1-score'])
        f1_positives.append(report[str(le.transform([class_labels[2]])[0])]['f1-score'])

        accuracies.append(acc)
        f1_macros.append(f1_macro)
        roc_aucs.append(roc_auc)

        print(f"  Fold {fold+1}: Accuracy={acc:.3f}, Macro F1={f1_macro:.3f}, ROC AUC={roc_auc:.3f}, "
              f"Neg F1={report[str(le.transform([class_labels[0]])[0])]['f1-score']:.3f}, "
              f"Neu F1={report[str(le.transform([class_labels[1]])[0])]['f1-score']:.3f}, "
              f"Pos F1={report[str(le.transform([class_labels[2]])[0])]['f1-score']:.3f}")

    print(f"\n--- Average Results for {embedding_name} with target_n={target_n} ---")
    print(f"  Average Accuracy: {np.mean(accuracies):.3f} (Std: {np.std(accuracies):.3f})")
    print(f"  Average Macro F1: {np.mean(f1_macros):.3f} (Std: {np.std(f1_macros):.3f})")
    print(f"  Average ROC AUC: {np.mean(roc_aucs):.3f} (Std: {np.std(roc_aucs):.3f})")
    print(f"  Average Negative F1: {np.mean(f1_negatives):.3f} (Std: {np.std(f1_negatives):.3f})")
    print(f"  Average Neutral F1: {np.mean(f1_neutrals):.3f} (Std: {np.std(f1_neutrals):.3f})")
    print(f"  Average Positive F1: {np.mean(f1_positives):.3f} (Std: {np.std(f1_positives):.3f})")
    print(f"--- Finished {model_type} Cross-Validation for {embedding_name} ---")

    all_y_true_concat = np.concatenate(all_y_true)
    all_y_pred_concat = np.concatenate(all_y_pred)

    return {
        'accuracies': accuracies,
        'f1_macros': f1_macros,
        'roc_aucs': roc_aucs,
        'f1_negatives': f1_negatives,
        'f1_neutrals': f1_neutrals,
        'f1_positives': f1_positives,
        'average_accuracy': np.mean(accuracies),
        'average_f1_macro': np.mean(f1_macros),
        'average_roc_auc': np.mean(roc_aucs),
        'average_f1_negative': np.mean(f1_negatives),
        'average_f1_neutral': np.mean(f1_neutrals),
        'average_f1_positive': np.mean(f1_positives),
        'all_y_true': all_y_true_concat,
        'all_y_pred': all_y_pred_concat,
        'le': le}

In [ ]:
#run the models now
def run_single_model_experiment(model_type, X_train, y_train, X_test, y_test, embedding_name, target_n, random_state=42):
    print(f"--- Running {model_type} experiment for {embedding_name} with target_n={target_n} ---")

    # Determine y_train and y_test to be used for model training/evaluation.
    # For XGBoost, these must be numeric. For others, they can be strings.
    y_train_for_model = y_train
    y_test_for_model = y_test

    # Ensure y_test is numerically encoded for roc_auc_score if it's not already, and if model_type is not XGBoost
    # For XGBoost, y_train/y_test are already numeric from previous steps (cell Q_MWeT6Y3Se4 or explicit encoding within this function).
    # For SVM/RF, y_train/y_test are categorical strings. They need to be numerically encoded for roc_auc_score.
    original_y_test = y_test # Keep original for classification_report if needed with string labels

    if not np.issubdtype(y_train.dtype, np.number) or model_type != 'XGBoost':
        le = LabelEncoder()
        # Fit encoder on combined train+test labels to ensure all labels are known
        all_labels = np.unique(np.concatenate((y_train, y_test)))
        le.fit(all_labels)
        y_train_encoded = le.transform(y_train)
        y_test_encoded = le.transform(y_test)

        y_train_for_model = y_train_encoded
        y_test_for_model = y_test_encoded


    # Apply resampling to the training data.
    X_bal, y_bal = resample_to_target(X_train, y_train_for_model, target_n, random_state=random_state)

    model = None
    if model_type == 'SVM':
        model = SVC(
            kernel='rbf',
            C=1e5,
            gamma='scale',
            decision_function_shape='ovo',
            probability=True # Required for predict_proba
        )
    elif model_type == 'Random Forest':
        model = RandomForestClassifier(
            n_estimators=100,
            random_state=random_state
        )
    elif model_type == 'XGBoost':
        # num_class should be based on the unique classes present in the balanced training data
        model = XGBClassifier(
            objective="multi:softprob",
            num_class=len(np.unique(y_train_for_model)), # Use actual number of classes after encoding
            n_estimators=200,
            learning_rate=0.1,
            max_depth=6,
            random_state=random_state,
            eval_metric="mlogloss"
        )
    else:
        raise ValueError("Invalid model_type specified")

    # Fit the model with the balanced training data
    model.fit(X_bal, y_bal)

    # Make predictions on the test data
    y_pred = model.predict(X_test)

    # Get prediction probabilities for ROC AUC
    y_proba = model.predict_proba(X_test)

    # Calculate evaluation metrics using the appropriate y_test_for_model for consistency
    acc = accuracy_score(y_test_for_model, y_pred)
    f1_macro = f1_score(y_test_for_model, y_pred, average='macro', zero_division=0)
    roc_auc = roc_auc_score(y_test_for_model, y_proba, multi_class='ovr', average='macro')

    # If original_y_test was used for classification report, use that, otherwise use encoded
    report = classification_report(original_y_test if 'original_y_test' in locals() else y_test_for_model, y_pred, output_dict=True, zero_division=0)

    return acc, f1_macro, roc_auc, report

In [ ]:
results_cv_balanced = {}
target_n_sweetspot = 171

# SVM
print("\n--- Running SVM with optimal resampling size ---")
svm_e2v_cv_results = run_cv_experiment('SVM', Xe2v, ye2v, "emoji2vec", target_n=target_n_sweetspot)
results_cv_balanced['SVM_emoji2vec'] = svm_e2v_cv_results

svm_emo_cv_results = run_cv_experiment('SVM', Xemo, yemo, "emojional", target_n=target_n_sweetspot)
results_cv_balanced['SVM_emojional'] = svm_emo_cv_results

# Random Forest
print("\n--- Running Random Forest with optimal resampling size ---")
rf_e2v_cv_results = run_cv_experiment('Random Forest', Xe2v, ye2v, "emoji2vec", target_n=target_n_sweetspot)
results_cv_balanced['RF_emoji2vec'] = rf_e2v_cv_results

rf_emo_cv_results = run_cv_experiment('Random Forest', Xemo, yemo, "emojional", target_n=target_n_sweetspot)
results_cv_balanced['RF_emojional'] = rf_emo_cv_results

# XGBoost
print("\n--- Running XGBoost with optimal resampling size ---")
xgb_e2v_cv_results = run_cv_experiment('XGBoost', Xe2v, ye2v, "emoji2vec", target_n=target_n_sweetspot)
results_cv_balanced['XGB_emoji2vec'] = xgb_e2v_cv_results

xgb_emo_cv_results = run_cv_experiment('XGBoost', Xemo, yemo, "emojional", target_n=target_n_sweetspot)
results_cv_balanced['XGB_emojional'] = xgb_emo_cv_results

print("\n--- All balanced cross-validation experiments completed ---")

In [ ]:
data_for_df_cv = []
label_names = {0: 'Negative', 1: 'Neutral', 2: 'Positive'}

for key, val in results_cv_balanced.items():
    model_embedding = key.split('_')
    model_name = model_embedding[0]
    embedding_name = model_embedding[1]

    # Use the label encoder from the results to get original class names
    le_obj = val['le']
    class_labels_original = le_obj.inverse_transform([0, 1, 2])

    row = {
        'Model': model_name,
        'Embedding': embedding_name,
        'Average Accuracy': val['average_accuracy'],
        'Average Macro F1': val['average_f1_macro'],
        'Average ROC AUC': val['average_roc_auc'],
        f'Average F1_{class_labels_original[0]}': val['average_f1_negative'],
        f'Average F1_{class_labels_original[1]}': val['average_f1_neutral'],
        f'Average F1_{class_labels_original[2]}': val['average_f1_positive']
    }
    data_for_df_cv.append(row)

results_cv_df = pd.DataFrame(data_for_df_cv)
display(results_cv_df)

# --- T-Tests for Balanced Cross-Validation Results ---
print("\n*** Comparing Models within emoji2vec Embeddings (Balanced CV) ***")
run_t_tests(results_cv_balanced['SVM_emoji2vec'], results_cv_balanced['RF_emoji2vec'], "SVM", "Random Forest", "emoji2vec Balanced CV")
run_t_tests(results_cv_balanced['SVM_emoji2vec'], results_cv_balanced['XGB_emoji2vec'], "SVM", "XGBoost", "emoji2vec Balanced CV")
run_t_tests(results_cv_balanced['RF_emoji2vec'], results_cv_balanced['XGB_emoji2vec'], "Random Forest", "XGBoost", "emoji2vec Balanced CV")

print("\n*** Comparing Models within emojional Embeddings (Balanced CV) ***")
run_t_tests(results_cv_balanced['SVM_emojional'], results_cv_balanced['RF_emojional'], "SVM", "Random Forest", "emojional Balanced CV")
run_t_tests(results_cv_balanced['SVM_emojional'], results_cv_balanced['XGB_emojional'], "SVM", "XGBoost", "emojional Balanced CV")
run_t_tests(results_cv_balanced['RF_emojional'], results_cv_balanced['XGB_emojional'], "Random Forest", "XGBoost", "emojional Balanced CV")

print("\n*** Comparing Embeddings for each Model (Balanced CV) ***")
run_t_tests(results_cv_balanced['SVM_emoji2vec'], results_cv_balanced['SVM_emojional'], "emoji2vec", "emojional", "SVM Balanced CV")
run_t_tests(results_cv_balanced['RF_emoji2vec'], results_cv_balanced['RF_emojional'], "emoji2vec", "emojional", "Random Forest Balanced CV")
run_t_tests(results_cv_balanced['XGB_emoji2vec'], results_cv_balanced['XGB_emojional'], "emoji2vec", "emojional", "XGBoost Balanced CV")

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

# Define class labels for display
class_labels_display = ['Negative', 'Neutral', 'Positive']

# Set up a 2x3 grid of subplots for the six confusion matrices
fig, axes = plt.subplots(2, 3, figsize=(24, 12)) # Adjusted figure size for better readability
plt.rcParams.update({'font.size': 10}) # Adjust font size for better readability

# Define model and embedding types for iteration, matching dictionary keys
model_prefixes = ['SVM', 'RF', 'XGB']
embedding_suffixes = ['emoji2vec', 'emojional']

# Iterate through embedding types (rows) and model types (columns)
for row_idx, embedding_name in enumerate(embedding_suffixes):
    for col_idx, model_name_prefix in enumerate(model_prefixes):
        # Construct the key for results_cv_balanced dictionary
        key = f'{model_name_prefix}_{embedding_name}'

        # Retrieve results for the current model-embedding combination
        current_results = results_cv_balanced[key]

        # Extract true and predicted labels
        all_y_true = current_results['all_y_true']
        all_y_pred = current_results['all_y_pred']

        # Create and plot Confusion Matrix
        ConfusionMatrixDisplay.from_predictions(
            all_y_true,
            all_y_pred,
            display_labels=class_labels_display, # Use the predefined class labels
            cmap=plt.cm.Blues,
            ax=axes[row_idx, col_idx]
        )
        # Set an appropriate title for each subplot
        axes[row_idx, col_idx].set_title(f'Confusion Matrix: {model_name_prefix} with {embedding_name} embeddings (CV)')

# Adjust the layout of the plots to prevent overlapping titles and labels
plt.tight_layout()
# Display the plots
plt.show()

## Bootstrapping

In [ ]:
def run_bootstrap_experiment(model_type, X_original, y_original, embedding_name, n_iterations=100, random_state=42, target_n=None):
    print(f"--- Running Bootstrapping for {model_type} with {embedding_name} (target_n={target_n})---")

    accuracies = []
    f1_macros = []
    f1_negatives = []
    f1_neutrals = []
    f1_positives = []

    # Ensure y is encoded for consistency with models, especially XGBoost
    le = LabelEncoder()
    y_encoded = le.fit_transform(y_original)
    num_classes = len(np.unique(y_encoded))
    class_labels = sorted(le.classes_)

    for i in range(n_iterations):
        # Set a seed for reproducibility within each iteration
        np.random.seed(random_state + i)

        # Generate stratified bootstrap indices by sampling with replacement within each class
        stratified_bootstrap_indices = []
        unique_classes = np.unique(y_encoded)
        for c in unique_classes:
            class_indices = np.where(y_encoded == c)[0]
            n_samples_for_class = len(class_indices)
            # Sample with replacement from indices of the current class
            resampled_class_indices = np.random.choice(class_indices, size=n_samples_for_class, replace=False)
            stratified_bootstrap_indices.extend(resampled_class_indices)

        # Convert to numpy array and shuffle to mix samples from different classes
        stratified_bootstrap_indices = np.array(stratified_bootstrap_indices)
        np.random.shuffle(stratified_bootstrap_indices) # Shuffle to mix classes

        X_boot_sample = X_original[stratified_bootstrap_indices]
        y_boot_sample = y_encoded[stratified_bootstrap_indices]

        # Split the bootstrapped data into training and testing sets (stratified split here)
        X_train_boot, X_test_boot, y_train_boot, y_test_boot = train_test_split(
            X_boot_sample, y_boot_sample, test_size=0.20, random_state=random_state + i, stratify=y_boot_sample)

        # Scale features
        scaler = StandardScaler()
        X_train_boot_scaled = scaler.fit_transform(X_train_boot)
        X_test_boot_scaled = scaler.transform(X_test_boot)

        # Apply resampling to the training data if target_n is provided
        X_final_train, y_final_train = X_train_boot_scaled, y_train_boot
        if target_n is not None:
            X_final_train, y_final_train = resample_to_target(X_train_boot_scaled, y_train_boot, target_n)

        model = None
        if model_type == 'SVM':
            model = SVC(
                kernel='rbf',
                C=1e5,
                gamma='scale',
                decision_function_shape='ovo',
                probability=True,
                random_state=random_state
            )
        elif model_type == 'Random Forest':
            model = RandomForestClassifier(
                n_estimators=100,
                max_depth=6,
                min_samples_leaf=5,
                random_state=random_state
            )
        elif model_type == 'XGBoost':
            model = XGBClassifier(
                objective="multi:softprob",
                num_class=num_classes,
                n_estimators=200,
                learning_rate=0.1,
                max_depth=6,
                random_state=random_state,
                eval_metric="mlogloss"
            )
        else:
            raise ValueError("Invalid model_type specified.")

        model.fit(X_final_train, y_final_train)
        y_pred_boot = model.predict(X_test_boot_scaled)
        y_proba_boot = model.predict_proba(X_test_boot_scaled)

        acc = accuracy_score(y_test_boot, y_pred_boot)
        f1_macro = f1_score(y_test_boot, y_pred_boot, average='macro', zero_division=0)

        report = classification_report(y_test_boot, y_pred_boot, output_dict=True, zero_division=0)

        accuracies.append(acc)
        f1_macros.append(f1_macro)
        f1_negatives.append(report[str(le.transform([class_labels[0]])[0])]['f1-score'])
        f1_neutrals.append(report[str(le.transform([class_labels[1]])[0])]['f1-score'])
        f1_positives.append(report[str(le.transform([class_labels[2]])[0])]['f1-score'])

        if (i + 1) % 10 == 0: # Print progress every 10 iterations
            print(f"  Iteration {i+1}/{n_iterations}: Acc={acc:.3f}, Macro F1={f1_macro:.3f}")

    print(f"--- Finished Bootstrapping for {model_type} with {embedding_name} ---")
    return {
        'accuracies': np.array(accuracies),
        'f1_macros': np.array(f1_macros),
        'f1_negatives': np.array(f1_negatives),
        'f1_neutrals': np.array(f1_neutrals),
        'f1_positives': np.array(f1_positives),
        'le': le
    }

In [ ]:
# Number of bootstrap iterations
N_BOOTSTRAP_ITERATIONS = 100
TARGET_N_BOOTSTRAP = 171

# Dictionary to store bootstrap results for balanced data
bootstrap_results_balanced = {}

# --- emoji2vec Embeddings ---
print("\n*** Running Bootstrapping for emoji2vec Embeddings with balanced data ***")
bootstrap_results_balanced['SVM_e2v'] = run_bootstrap_experiment('SVM', Xe2v, ye2v, "emoji2vec", n_iterations=N_BOOTSTRAP_ITERATIONS, target_n=TARGET_N_BOOTSTRAP)
bootstrap_results_balanced['RF_e2v'] = run_bootstrap_experiment('Random Forest', Xe2v, ye2v, "emoji2vec", n_iterations=N_BOOTSTRAP_ITERATIONS, target_n=TARGET_N_BOOTSTRAP)
bootstrap_results_balanced['XGB_e2v'] = run_bootstrap_experiment('XGBoost', Xe2v, ye2v, "emoji2vec", n_iterations=N_BOOTSTRAP_ITERATIONS, target_n=TARGET_N_BOOTSTRAP)

# --- emojional Embeddings ---
print("\n*** Running Bootstrapping for emojional Embeddings with balanced data ***")
bootstrap_results_balanced['SVM_emo'] = run_bootstrap_experiment('SVM', Xemo, yemo, "emojional", n_iterations=N_BOOTSTRAP_ITERATIONS, target_n=TARGET_N_BOOTSTRAP)
bootstrap_results_balanced['RF_emo'] = run_bootstrap_experiment('Random Forest', Xemo, yemo, "emojional", n_iterations=N_BOOTSTRAP_ITERATIONS, target_n=TARGET_N_BOOTSTRAP)
bootstrap_results_balanced['XGB_emo'] = run_bootstrap_experiment('XGBoost', Xemo, yemo, "emojional", n_iterations=N_BOOTSTRAP_ITERATIONS, target_n=TARGET_N_BOOTSTRAP)

print("\n*** All Balanced Bootstrapping experiments completed ***")

In [ ]:
# Re-calculate plot_df for balanced bootstrap results
plot_data_balanced = []

for key, res in bootstrap_results_balanced.items():
    model_name = key.split('_')[0]
    embedding_name = 'emoji2vec' if 'e2v' in key else 'emojional'

    # Accuracy
    mean_acc = np.mean(res['accuracies'])
    lower_acc, upper_acc = calculate_confidence_interval(res['accuracies'])
    plot_data_balanced.append({
        'Metric': 'Accuracy',
        'Model': model_name,
        'Embedding': embedding_name,
        'Mean': mean_acc,
        'Lower_CI': lower_acc,
        'Upper_CI': upper_acc
    })

    # Macro F1
    mean_f1 = np.mean(res['f1_macros'])
    lower_f1, upper_f1 = calculate_confidence_interval(res['f1_macros'])
    plot_data_balanced.append({
        'Metric': 'Macro F1',
        'Model': model_name,
        'Embedding': embedding_name,
        'Mean': mean_f1,
        'Lower_CI': lower_f1,
        'Upper_CI': upper_f1
    })

plot_df_balanced = pd.DataFrame(plot_data_balanced)
display(plot_df_balanced)

In [ ]:
metrics_balanced = plot_df_balanced['Metric'].unique()

for metric in metrics_balanced:
    df_subset = plot_df_balanced[plot_df_balanced['Metric'] == metric]

    models = df_subset['Model'].unique()
    embeddings = ["emoji2vec", "emojional"]
    y_pos = np.arange(len(models))
    offset = 0.2

    plt.figure(figsize=(8, 5))

    for i, emb in enumerate(embeddings):
        emb_data = df_subset[df_subset['Embedding'] == emb]

        means = emb_data['Mean'].values
        lower = emb_data['Lower_CI'].values
        upper = emb_data['Upper_CI'].values

        # Adjust y_positions for each embedding group
        y_positions = y_pos + (len(embeddings)/2 - i - 0.5) * offset

        # Horizontal CI lines
        for j in range(len(means)):
            plt.hlines(y=y_positions[j], xmin=lower[j], xmax=upper[j], color=plt.cm.Paired(i))

        # Mean dots
        plt.scatter(means, y_positions, label=emb, s=60, color=plt.cm.Paired(i))

    plt.yticks(y_pos, models)
    plt.xlabel(metric)
    plt.ylabel("Model")
    plt.title(f"{metric} with 95% Confidence Intervals (Balanced Data)")

    plt.xlim(0, 1)
    plt.grid(False)

    if metric == "Accuracy":
        plt.axvline(0.3333, linestyle='--', linewidth=1, color='gray', label='Baseline Accuracy')
    elif metric == "Macro F1":
        plt.axvline(0.3333, linestyle='--', linewidth=1, color='gray', label='Baseline Macro F1')

    plt.legend(title="Embedding", loc="upper left")
    plt.tight_layout()
    plt.show()